In [0]:
dbutils.widgets.text("batch", "0" , "Batch number to load")
batch_num = dbutils.widgets.get("batch") 

staging_root = f"/Volumes/dev1/stepright1/staging/batch_{batch_num}"
landing_root = "/Volumes/dev1/stepright1/landing1"

print(f" loading batch_{batch_num} from {staging_root} into {landing_root}" )
print(f"{staging_root}")

In [0]:
subfolders = [
    "orders_cdc",
    "order_items_cdc",
    "customers_cdc",
    "products",
    "categories",
    "clickstream",
    "inventory"
]

for folder in subfolders:
    src = f"{staging_root}/{folder}"
    dst = f"{landing_root}/{folder}"

    try:
        dbutils.fs.cp(src, dst, recurse=True)
        print(f" copied from {src} -> {dst}")
              
    except Exception as e:
    
           print(f"skipped {folder} (nothing staged for this batch:) {e}")

    


In [0]:
csv_sources = ["products", "categories", "inventory"]
json_sources = ["orders_cdc", "order_items_cdc", "customers_cdc", "clickstream"]

print(f"{'source':<20}{'format':<8}{'row_count'}")
print('-' *45)

for folder in csv_sources:
    path = f"{landing_root}/{folder}"

    try:
        count = spark.read.option("header" , "true").csv(path).count()
    except Exception:
        count = 0

    print(f"{folder:<20}{'csv:':<8}{count}")

for folder in json_sources:
    path = f"{landing_root}/{folder}"

    try:
        count = spark.read.json(path).count()

    except Exception:
        count = 0

    print(f"{folder:<20}{'json:':<8}{count}")

